In [ ]:
from selenium.webdriver.common.action_chains import ActionChains
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from datetime import datetime, timedelta
import undetected_chromedriver as uc
import pandas as pd
import random
import time

# ================= DATE =================
today = datetime.now()
tomorrow = today + timedelta(days=1)

checkin_year = today.year
checkin_month = today.month
checkin_day = today.day

checkout_year = tomorrow.year
checkout_month = tomorrow.month
checkout_day = tomorrow.day

# ================= CITIES =================
egypt_governorates = [
    "Alexandria",
    "Giza",
    "Cairo",
    "Luxor",
    "Red Sea",
    "Faiyum",
    "Qena",
    "Suez",
    "Ismailia",
    "Asyut"
]
c=0
list_places = []

# ================= DRIVER =================
options = Options()
options.add_argument("--disable-blink-features=AutomationControlled")

driver = uc.Chrome(options=options, version_main=146)
wait = WebDriverWait(driver, 15)

print("🚀 Starting Scraper...")

# ================= LOOP =================
for city in egypt_governorates:
    print(f"\n=====================================")
    print(f"🏙️ Now Scraping: {city}")
    print(f"=======================================")

    try:
        
        url = f"https://www.booking.com/searchresults.html?ss={city}" \
              f"&checkin_year={checkin_year}&checkin_month={checkin_month}&checkin_monthday={checkin_day}" \
              f"&checkout_year={checkout_year}&checkout_month={checkout_month}&checkout_monthday={checkout_day}"

        driver.get(url)
        if c == 0 :
            time.sleep(random.uniform(3, 5))
            ActionChains(driver).move_by_offset(10, 10).click().perform()
            c+=1


        # wait for results
        wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, '[data-testid="property-card"]')))

        places = driver.find_elements(By.CSS_SELECTOR, '[data-testid="property-card"]')

        print(f"--- Found {len(places)} places in {city}. Taking top 5 ---")

        for place in places[:5]:
            try:
                title = place.find_element(By.CSS_SELECTOR, '[data-testid="title"]').text.strip()
            except:
                title = 'No Title'

            try:
                address = place.find_element(By.CSS_SELECTOR, '[data-testid="address-link"]').text.strip()
            except:
                address = 'No Address'

            try:
                rating = place.find_element(By.CSS_SELECTOR, '[data-testid="review-score"]').text.strip()
                clean_rating = rating.replace('\n', ' | ')
            except:
                clean_rating = 'No Rating'

            try:
                price = place.find_element(By.CSS_SELECTOR, '[data-testid="price-and-discounted-price"]').text.strip()
            except:
                price = 'No Price'

            try:
                link = place.find_element(By.CSS_SELECTOR, 'h3 a').get_attribute("href")
            except:
                link = 'No Link'

            try:
                room_box = place.find_element(By.CLASS_NAME, "fff1944c52")
                clean_room_info = room_box.text.replace('\n', ' | ')
            except:
                clean_room_info = 'No Room Info'

            if title != 'No Title':
                list_places.append({
                    'Place_Name': title,
                    'Rating': clean_rating,
                    'Price': price,
                    'Address': address,
                    'Link': link,
                    'Room_Info': clean_room_info
                })

        time.sleep(random.uniform(2, 4))

    except Exception as e:
        print(f"Skipping {city} For Error: {e}")
        continue

driver.quit()

print(f"\nTotal scraped: {len(list_places)}")

if len(list_places) > 0:
    df = pd.DataFrame(list_places)
    df.to_csv('Egypt_All_Governorates_Hotels.csv', index=False, encoding='utf-8-sig')
    print("Data saved successfully as Egypt_All_Governorates_Hotels.csv")
else:
    print("No data scraped!")